[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/masanorihirano/pams/blob/main/examples/invest_div.ipynb)

In [ ]:
# Please remove comment-out if necessary
#! pip install pams matplotlib

# Regulation for investment diversification

This example is a port of the InvestDiv sample of plhamJ, which is based on
Nozaki, Mizuta and Yagi (2016) "Investigation of the rule for investment diversification at the time of a market crash
using an artificial market" (in Japanese).

There are two markets, Market-1 and Market-2, and the fundamental price of Market-1 falls by 10% at the beginning of
the second session. Half of the agents are `FCNAgent`s, and the other half are `InvestDivFCNAgent`s, which trade under
the regulation for investment diversification:

- An `InvestDivFCNAgent` calculates its net asset value, i.e., its cash amount plus the values of its assets in the
  markets.
- If the sum of the absolute asset values is larger than `leverageRatio` times the net asset value, it submits no
  orders.
- Otherwise, it makes the orders of `FCNAgent`. The order for a market is submitted as it is if the absolute asset
  value in the market is at most `diversityRatio` times the net asset value. If not, it is replaced by a market order
  of one unit that reduces the position in the market.

This example differs from the plhamJ sample as follows:

- `marginType` is `fixed` instead of `normal`. With the normal margin type, `FCNAgent` of PAMS decides whether an
  order is a buy or a sell order by comparing the expected price with the market price, while plhamJ compares it with
  the order price, and the prices diverge in PAMS.
- `maxNormalOrders` is 2, i.e., the number of markets, which is the default value in plhamJ.
- The market order of the regulation is submitted only if it can be executed at once, i.e., the market executes
  orders and the order book on the other side is not empty. In plhamJ, a market order that cannot be executed at once
  stays in the order book with a NaN price, which no limit order matches, until it expires. In PAMS, it would be
  executed later at the price of the next order on the other side.
- `leverageRatio` is used as configured. plhamJ always uses 1.0 because a local variable shadows the configured value,
  but it makes no difference here because the configured value is 1.0.

In [ ]:
import copy
import random
from typing import Any
from typing import Dict
from typing import List
from typing import Optional
from typing import Union

import matplotlib.pyplot as plt
import numpy as np

from pams import Simulator
from pams.agents import FCNAgent
from pams.logs import Logger
from pams.logs import OrderLog
from pams.logs.market_step_loggers import MarketStepSaver
from pams.market import Market
from pams.order import MARKET_ORDER
from pams.order import Cancel
from pams.order import Order
from pams.runners import SequentialRunner

In [ ]:
class InvestDivFCNAgent(FCNAgent):
    """FCN agent under the regulation for investment diversification."""

    def __init__(
        self,
        agent_id: int,
        prng: random.Random,
        simulator: Simulator,
        name: str,
        logger: Optional[Logger] = None,
    ) -> None:
        super().__init__(agent_id=agent_id, prng=prng, simulator=simulator, name=name, logger=logger)
        self.leverage_ratio: float = 1.0
        self.diversity_ratio: float = 1.0

    def setup(self, settings: Dict[str, Any], accessible_markets_ids: List[int], *args: Any, **kwargs: Any) -> None:
        super().setup(settings, accessible_markets_ids, *args, **kwargs)
        self.leverage_ratio = float(settings["leverageRatio"])
        self.diversity_ratio = float(settings["diversityRatio"])

    def get_asset_value(self, market: Market) -> float:
        return market.get_market_price() * self.get_asset_volume(market_id=market.market_id)

    def is_market_order_executable(self, market: Market, is_buy: bool) -> bool:
        if not market.is_running:
            return False
        return len(market.sell_order_book if is_buy else market.buy_order_book) > 0

    def submit_orders(self, markets: List[Market]) -> List[Union[Order, Cancel]]:
        accessible_markets = [market for market in markets if self.is_market_accessible(market_id=market.market_id)]
        net_asset_value = self.get_cash_amount()
        total_asset_value_abs = 0.0
        for market in accessible_markets:
            asset_value = self.get_asset_value(market=market)
            net_asset_value += asset_value
            total_asset_value_abs += abs(asset_value)

        orders: List[Union[Order, Cancel]] = []
        if total_asset_value_abs > self.leverage_ratio * net_asset_value:
            return orders

        id2market = {market.market_id: market for market in accessible_markets}
        for order in super().submit_orders(markets=accessible_markets):
            market = id2market[order.market_id]
            if abs(self.get_asset_value(market=market)) <= self.diversity_ratio * net_asset_value:
                orders.append(order)
                continue
            # replace the order with a market order reducing the position by one unit if it can be executed at once.
            asset_volume = self.get_asset_volume(market_id=market.market_id)
            if asset_volume != 0 and self.is_market_order_executable(market=market, is_buy=asset_volume < 0):
                orders.append(
                    Order(
                        agent_id=self.agent_id,
                        market_id=market.market_id,
                        is_buy=asset_volume < 0,
                        kind=MARKET_ORDER,
                        volume=1,
                        ttl=10,
                    )
                )
        return orders

The saver below also saves the market orders. Only the regulation submits market orders in this simulation.

In [ ]:
class InvestDivSaver(MarketStepSaver):
    def __init__(self) -> None:
        super().__init__()
        self.market_order_logs: List[Dict[str, Any]] = []

    def process_order_log(self, log: OrderLog) -> None:
        if log.kind == MARKET_ORDER:
            self.market_order_logs.append(
                {"market_time": log.time, "market_id": log.market_id, "is_buy": log.is_buy}
            )

In [ ]:
config = {
	"simulation": {
		"markets": ["Market-1", "Market-2"],
		"agents": ["InvestDivFCNAgents", "FCNAgents"],
		"sessions": [
			{	"sessionName": 0,
				"iterationSteps": 100,
				"withOrderPlacement": True,
				"withOrderExecution": False,
				"withPrint": True,
				"maxNormalOrders": 2, "MEMO": "The same number as #markets"
			},
			{	"sessionName": 1,
				"iterationSteps": 3000,
				"withOrderPlacement": True,
				"withOrderExecution": True,
				"withPrint": True,
				"maxNormalOrders": 2, "MEMO": "The same number as #markets",
				"events": ["FundamentalPriceShock"]
			}
		]
	},

	"FundamentalPriceShock": {
		"class": "FundamentalPriceShock",
		"target": "Market-1",
		"triggerTime": 0,    "MEMO": "At the beginning of the session 2",
		"priceChangeRate": -0.1,    "MEMO": "Sign: negative for down; positive for up; zero for no change",
		"enabled": True
	},

	"Market-1": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},

	"Market-2": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},

	"FCNAgents": {
		"class": "FCNAgent",
		"numAgents": 100,

		"MEMO": "Agent class",
		"markets": ["Market-1", "Market-2"],
		"assetVolume": 50,
		"cashAmount": 10000,

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [2.0]},
		"chartWeight": {"expon": [1.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 200],
		"marginType": "fixed",
		"orderMargin": [0.0, 0.1]
	},

	"InvestDivFCNAgents": {
		"class": "InvestDivFCNAgent",
		"numAgents": 100,

		"MEMO": "Agent class",
		"markets": ["Market-1", "Market-2"],
		"assetVolume": [10, 50],
		"cashAmount": [1000, 10000],

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [2.0]},
		"chartWeight": {"expon": [1.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 200],
		"marginType": "fixed",
		"orderMargin": [0.0, 0.1],

		"MEMO": "InvestDivFCNAgent class",
		"leverageRatio": 1.0,
		"diversityRatio": 0.4
	}
}

Without the regulation, the `InvestDivFCNAgents` are replaced by `FCNAgent`s with the same parameters.
`InvestDivFCNAgent` uses no additional random numbers, so the simulations with the same seed differ only by the
regulation.

In [ ]:
config_without_regulation = copy.deepcopy(config)
config_without_regulation["InvestDivFCNAgents"]["class"] = "FCNAgent"
configs = {"with regulation": config, "without regulation": config_without_regulation}


def run(config: Dict[str, Any], seed: int) -> InvestDivSaver:
    saver = InvestDivSaver()
    runner = SequentialRunner(
        settings=config,
        prng=random.Random(seed),
        logger=saver,
    )
    runner.class_register(cls=InvestDivFCNAgent)
    runner.main()
    return saver


def get_prices(saver: InvestDivSaver, key: str) -> Dict[str, np.ndarray]:
    return {
        market_name: np.asarray([log[key] for log in saver.market_step_logs if log["market_name"] == market_name])
        for market_name in ["Market-1", "Market-2"]
    }


def plot_prices(market_prices: Dict[str, np.ndarray], fundamental_prices: Dict[str, np.ndarray], title: str) -> None:
    plt.figure()
    plt.plot(market_prices["Market-1"], color="red", label="Market-1")
    plt.plot(market_prices["Market-2"], color="green", label="Market-2")
    plt.plot(fundamental_prices["Market-1"], color="black")
    plt.plot(fundamental_prices["Market-2"], color="black")
    plt.xlabel("ticks")
    plt.ylabel("market price")
    plt.ylim([200, 310])
    plt.title(title)
    plt.legend()
    plt.show()

## A simulation with and without the regulation

In [ ]:
savers = {case: run(config=case_config, seed=42) for case, case_config in configs.items()}

In [ ]:
for case, saver in savers.items():
    plot_prices(
        market_prices=get_prices(saver=saver, key="market_price"),
        fundamental_prices=get_prices(saver=saver, key="fundamental_price"),
        title=case,
    )

The market orders of the regulation reduce the long positions of the regulated agents, i.e., they are sell orders.
They are submitted not only in Market-1 but also in Market-2, whose fundamental price does not change.

In [ ]:
saver = savers["with regulation"]
market_id2name = {log["market_id"]: log["market_name"] for log in saver.market_step_logs}
for market_id, market_name in market_id2name.items():
    logs = [log for log in saver.market_order_logs if log["market_id"] == market_id]
    n_buy = sum(log["is_buy"] for log in logs)
    print(f"{market_name}: {len(logs)} market orders ({n_buy} buy, {len(logs) - n_buy} sell)")

plt.figure()
plt.hist(
    [
        [log["market_time"] for log in saver.market_order_logs if log["market_id"] == market_id]
        for market_id in market_id2name.keys()
    ],
    bins=np.arange(0, 3101, 100),
    color=["red", "green"],
    label=list(market_id2name.values()),
)
plt.xlabel("ticks")
plt.ylabel("market orders by the regulation per 100 ticks")
plt.legend()
plt.show()

## Prices averaged over simulations

As `plot.R` of the plhamJ sample does, the market prices are averaged over simulations with different seeds.
Please add seeds for smoother results.
Note that the market prices diverge with some seeds, and the simulations fail with some seeds (with 1 of the 200 seeds
from 0 to 199 with the regulation, and with 15 of them without the regulation).
Also, the prices have short spikes, as the output of the plhamJ sample does, and the range of the y axis, which is the
same as `plot.R`, clips them.

In [ ]:
seeds = [42, 43, 44]
average_prices = {}
for case, case_config in configs.items():
    case_savers = [savers[case] if seed == 42 else run(config=case_config, seed=seed) for seed in seeds]
    prices = [get_prices(saver=case_saver, key="market_price") for case_saver in case_savers]
    average_prices[case] = {
        market_name: np.mean([price[market_name] for price in prices], axis=0) for market_name in ["Market-1", "Market-2"]
    }
    plot_prices(
        market_prices=average_prices[case],
        fundamental_prices=get_prices(saver=case_savers[0], key="fundamental_price"),
        title=f"{case} (average of {len(seeds)} simulations)",
    )

In these simulations, the price of Market-2 falls slightly with the regulation although its fundamental price does
not change, i.e., the regulation transfers the shock in Market-1 to Market-2.

In [ ]:
print("average market prices in the last 500 ticks")
for case, prices in average_prices.items():
    print(f"{case:>20}: " + ", ".join(f"{name} {price[-500:].mean():.1f}" for name, price in prices.items()))